# Notebook 07 — Resultados variados (justificacion de datos y generalizacion)

Este notebook reune comprobaciones empiricas complementarias al pipeline principal
(notebooks 03-05). Se organiza en **tres bloques independientes**:

- **7.0 Completitud del catalogo USGS en Filipinas** — por que no se anaden mas regiones
  via USGS (Mc demasiado alto fuera de zonas con redes locales densas).
- **7.1 STEAD vs USGS (California)** — por que el pronostico usa el catalogo USGS y no STEAD
  (completitud, homogeneidad temporal y cobertura).
- **7.2 Transferencia del Transformer a otras regiones** — se aplica el modelo entrenado en
  California+Alaska, en inferencia, a otra region, para medir si generaliza o si su AUC cae.

> Cada bloque es autonomo. El 7.0 y el 7.2 requieren conexion a internet (FDSN del USGS); el
> 7.2 necesita ademas los checkpoints `results/05_ckpt_seed*.pt` del Transformer.

In [ ]:
# Entorno + NB_TAG (derivado automaticamente del NOMBRE de este notebook)
import os

try:
    from google.colab import drive
    print("Entorno Colab detectado: Montando Google Drive...")
    drive.mount('/content/drive')
    import sys
    sys.path.append("/content/drive/MyDrive/VIU/TFM/code")
    sys.path.append("/content/drive/MyDrive/VIU/TFM/STEAD")
    sys.path.append("/content/drive/MyDrive/VIU/TFM/USGS")
    from google.colab import _message
    _meta = _message.blocking_request("get_ipynb")["ipynb"]["metadata"]
    _nb_name = _meta.get("colab", {}).get("name", "07_STEAD_vs_USGS.ipynb")
    NB_TAG = os.path.splitext(_nb_name)[0].split("_")[0]
except ImportError:
    try:
        NB_TAG = os.path.splitext(os.path.basename(__vsc_ipynb_file__))[0].split("_")[0]
    except NameError:
        NB_TAG = "07"
print(f"NB_TAG = {NB_TAG}")

## 7.0 Completitud del catalogo USGS fuera de EE.UU.: el caso de Filipinas

Antes de la comparacion STEAD vs USGS, una comprobacion complementaria que justifica
**por que el pipeline se cine a California + Alaska y no anade mas regiones activas**
(p. ej. Filipinas) usando el catalogo USGS.

El USGS (ANSS ComCat) es un catalogo **global**, pero su **completitud no es homogenea**:

- En California / Alaska integra redes locales densas (SCSN, NCSN, AEC) -> **Mc ~ 2.5**.
- En Filipinas depende de la **deteccion telesismica** de la red global -> **Mc mucho mas alto**.

Consecuencia: en Filipinas faltarian casi todos los eventos M2.5-4.5 (los microsismos que
alimentan las *features* de pronostico), aunque sobren objetivos M>=4.5. La celda siguiente
lo verifica descargando el catalogo USGS de Filipinas (2000-2024) del FDSN web service y
estimando su Mc (MAXC) y su b-value, para compararlos con el Mc=2.5 de California.

> **Resultado esperado:** Mc(Filipinas) ~ 4.5, frente a 2.5 en California. Es decir, el USGS
> NO cubre Filipinas con la finura que exige el metodo. Anadir la region daria mas positivos
> pero romperia la premisa de catalogo completo a baja magnitud; para hacerlo bien se
> necesitaria el catalogo local de **PHIVOLCS**, no el USGS.

In [ ]:
# =====================================================================
# 7.0  Completitud del catalogo USGS FUERA de EE.UU.: el caso de Filipinas
# ---------------------------------------------------------------------
# Comprobacion empirica de por que NO conviene anadir regiones como
# Filipinas al pipeline usando solo el catalogo USGS. En California/Alaska
# el USGS integra redes locales densas (SCSN/NCSN/AEC) -> Mc ~ 2.5. En
# Filipinas depende de la deteccion telesismica global -> Mc mucho mas alto,
# de modo que faltan los microsismos que necesitan las features.
# (Requiere internet: consulta el FDSN web service del USGS ano por ano.)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    NB_TAG
except NameError:
    NB_TAG = "07"
FIGURES_DIR = os.path.join("..", "results", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

PH_BOX = {"minlat": 4, "maxlat": 21, "minlon": 116, "maxlon": 128}  # Filipinas
PH_START, PH_END = 2000, 2024

def descargar_usgs(box, y0, y1, mmin=0):
    """Descarga el catalogo USGS (FDSN) ano por ano (evita el limite de 20000/peticion)."""
    frames = []
    for yr in range(y0, y1 + 1):
        url = ("https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv"
               f"&starttime={yr}-01-01&endtime={yr}-12-31"
               f"&minlatitude={box['minlat']}&maxlatitude={box['maxlat']}"
               f"&minlongitude={box['minlon']}&maxlongitude={box['maxlon']}"
               f"&minmagnitude={mmin}")
        try:
            frames.append(pd.read_csv(url))
        except Exception as e:
            print(f"  ano {yr}: fallo ({type(e).__name__})")
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

print("Descargando catalogo USGS de Filipinas (puede tardar ~1 min)...")
ph = descargar_usgs(PH_BOX, PH_START, PH_END, mmin=0).dropna(subset=["mag"])
m = ph["mag"].values
print(f"  eventos descargados: {len(m):,}  (rango M {m.min():.1f}-{m.max():.1f})")

def mc_maxc(mags, bw=0.1):
    bins = np.arange(np.floor(mags.min()*10)/10, np.ceil(mags.max()*10)/10 + bw, bw)
    c, e = np.histogram(mags, bins=bins)
    return float(e[int(c.argmax())] + bw / 2)

mc_ph = mc_maxc(m)
b_ph = np.log10(np.e) / (m[m >= mc_ph].mean() - (mc_ph - 0.05))   # estimador de Aki
MC_CALIFORNIA = 2.5

print("\n=== Completitud USGS: Filipinas vs California ===")
print(f"  Mc (MAXC) Filipinas = {mc_ph:.2f}   |   California = {MC_CALIFORNIA:.2f}")
print(f"  b-value Filipinas   = {b_ph:.2f}")
print(f"  eventos M>=2.5: {int((m>=2.5).sum()):,}   M>=4.0: {int((m>=4.0).sum()):,}   M>=4.5: {int((m>=4.5).sum()):,}")
print(f"  fraccion de eventos por debajo de M4.0: {100*(m<4.0).mean():.1f}%")

# --- Figura: distribucion frecuencia-magnitud de Filipinas con su Mc ---
mbins = np.arange(0, 8, 0.1)
cum = np.array([(m >= mb).sum() for mb in mbins])
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
axs[0].hist(m, bins=mbins, color="seagreen", alpha=0.75)
axs[0].axvline(mc_ph, color="black", ls="--", lw=1.5, label=f"Mc Filipinas = {mc_ph:.2f}")
axs[0].axvline(MC_CALIFORNIA, color="crimson", ls=":", lw=1.5, label=f"Mc California = {MC_CALIFORNIA}")
axs[0].set_xlabel("Magnitud"); axs[0].set_ylabel("n eventos")
axs[0].set_title("Filipinas (USGS): histograma de magnitudes", fontweight="bold")
axs[0].legend(); axs[0].grid(alpha=0.3)
axs[1].semilogy(mbins, cum, "o-", ms=3, color="seagreen", label="USGS Filipinas")
axs[1].axvline(mc_ph, color="black", ls="--", lw=1.5, label=f"Mc = {mc_ph:.2f}")
axs[1].axvline(MC_CALIFORNIA, color="crimson", ls=":", lw=1.5, label=f"Mc California = {MC_CALIFORNIA}")
axs[1].set_xlabel("Magnitud"); axs[1].set_ylabel("N acumulado (>= M)")
axs[1].set_title("Filipinas (USGS): Gutenberg-Richter", fontweight="bold")
axs[1].legend(); axs[1].grid(alpha=0.3)
plt.tight_layout()
p_ph = os.path.join(FIGURES_DIR, f"{NB_TAG}_filipinas_usgs_mc.png")
plt.savefig(p_ph, dpi=300, bbox_inches="tight"); plt.show()
print(f"Figura: {p_ph}")

print()
print("=" * 70)
print("CONCLUSION (por que NO se anaden regiones como Filipinas via USGS):")
print(f"  - El USGS en Filipinas es completo solo a Mc~{mc_ph:.1f} (vs 2.5 en California).")
print(f"  - El {100*(m<4.0).mean():.0f}% de los eventos ya esta por encima de M4.0: NO hay microsismos.")
print("  - Las features del pronostico (log_count, b-value dia a dia) exigen catalogo")
print("    completo a M2.5; con Mc~4.5 quedarian sesgadas o vacias.")
print("  - Habria targets (M>=4.5) de sobra, pero faltaria el catalogo de ENTRADA.")
print("  => Para Filipinas se necesitaria el catalogo LOCAL de PHIVOLCS, no el USGS.")
print("     Esto justifica cenirse a California+Alaska (redes densas, Mc bajo y homogeneo).")
print("=" * 70)

## 7.1 STEAD vs USGS (California): por que el pronostico usa USGS

STEAD es una **seleccion curada para entrenar pickers**, no un catalogo completo ni homogeneo
en el tiempo. Lo demostramos sobre **California**, comparando a **igual magnitud (M>=2.5)** el
catalogo de STEAD (deduplicado por evento) contra el del USGS:

1. **Completitud**: a M>=2.5, STEAD tiene MENOS eventos que USGS (le faltan eventos reales).
2. **Homogeneidad temporal**: los eventos/anio de STEAD varian ~100x de forma artificial.
3. **Cobertura**: STEAD termina en 2018 -> no cubre el periodo de test (2023-2024).

Las celdas siguientes cargan ambos catalogos y generan las dos figuras de la comparacion
(`07_stead_vs_usgs_cobertura.png` y `07_stead_vs_usgs_magnitud_mc.png`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Configuracion de la comparacion ---
REGION = {"minlat": 32.0, "maxlat": 42.0, "minlon": -124.0, "maxlon": -114.0}  # California
START = pd.Timestamp("2000-01-01", tz="UTC")
END   = pd.Timestamp("2018-12-31", tz="UTC")     # STEAD cubre hasta 2018
PERIODO = f"{START.year}-{END.year}"
MC_CMP = 2.5     # umbral comun para la comparacion JUSTA (mismo Mc que el pronostico)

RESULTS_DIR = os.path.join("..", "results")
FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)
STEAD_CSV = os.path.join("..", "STEAD", "features.csv")
USGS_CSV  = os.path.join("..", "USGS", "california_2000_2024_M2.5.csv")
print(f"Region: California  |  Periodo: {PERIODO}  |  comparacion justa a M>={MC_CMP}")

In [ ]:
# --- STEAD: earthquakes en California, deduplicado por evento ---
df_s = pd.read_csv(STEAD_CSV, low_memory=False)
print(f"STEAD CSV total filas (trazas): {len(df_s):,}")

df_s = df_s[df_s["trace_category"].astype(str).str.contains("earthquake", case=False, na=False)]
df_s["origin"] = pd.to_datetime(df_s["source_origin_time"], errors="coerce", utc=True)
msk = (df_s["source_latitude"].between(REGION["minlat"], REGION["maxlat"]) &
       df_s["source_longitude"].between(REGION["minlon"], REGION["maxlon"]) &
       df_s["origin"].ge(START) & df_s["origin"].le(END))
df_s = df_s[msk].copy()

ev_s = df_s.drop_duplicates(subset="source_id").copy()
ev_s["mag"] = pd.to_numeric(ev_s["source_magnitude"], errors="coerce")
ev_s = ev_s.dropna(subset=["mag"])
ev_s25 = ev_s[ev_s["mag"] >= MC_CMP]
print(f"  eventos unicos (todas magnitudes): {len(ev_s):,}   (STEAD baja hasta M~0.5)")
print(f"  eventos unicos M>={MC_CMP}:            {len(ev_s25):,}")

In [ ]:
# --- USGS: California, mismo periodo (ya es M>=2.5, solo earthquakes) ---
df_u = pd.read_csv(USGS_CSV)
df_u["time"] = pd.to_datetime(df_u["time"], errors="coerce", utc=True)
if "type" in df_u.columns:
    df_u = df_u[df_u["type"] == "earthquake"]
msk = (df_u["latitude"].between(REGION["minlat"], REGION["maxlat"]) &
       df_u["longitude"].between(REGION["minlon"], REGION["maxlon"]) &
       df_u["time"].ge(START) & df_u["time"].le(END))
ev_u = df_u[msk].dropna(subset=["mag"]).copy()
ev_u25 = ev_u[ev_u["mag"] >= MC_CMP]
print(f"USGS eventos M>={MC_CMP} en California {PERIODO}: {len(ev_u25):,}")

In [ ]:
# --- Comparativa 1: completitud + homogeneidad temporal (a M>=2.5) ---
print(f"=== Eventos M>={MC_CMP}, California {PERIODO} ===")
print(f"  STEAD: {len(ev_s25):>8,}")
print(f"  USGS:  {len(ev_u25):>8,}   ({len(ev_u25)/max(len(ev_s25),1):.1f}x mas que STEAD)")
print(f"  -> a igual magnitud USGS tiene mas: STEAD es INCOMPLETO.")

yr_s = ev_s25["origin"].dt.year.value_counts().sort_index()
yr_u = ev_u25["time"].dt.year.value_counts().sort_index()
# medida de (no) homogeneidad: ratio max/min de eventos por anio
homog = lambda v: v.max()/max(v.min(),1)
print(f"\nVariacion eventos/anio (max/min):  STEAD {homog(yr_s):.0f}x   USGS {homog(yr_u):.0f}x")
print(f"  -> STEAD oscila ~100x por anio = muestreo artificial, NO sismicidad real.")

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(yr_u.index - 0.2, yr_u.values, width=0.4, label=f"USGS ({len(ev_u25):,})", color="steelblue")
ax.bar(yr_s.index + 0.2, yr_s.values, width=0.4, label=f"STEAD ({len(ev_s25):,})", color="darkred")
ax.set_xlabel("Anio"); ax.set_ylabel(f"n eventos M>={MC_CMP}")
ax.set_title(f"Cobertura temporal a M>={MC_CMP}: USGS (homogeneo) vs STEAD (irregular, acaba 2018)",
             fontweight="bold", fontsize=11)
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
p1 = os.path.join(FIGURES_DIR, f"{NB_TAG}_stead_vs_usgs_cobertura.png")
plt.savefig(p1, dpi=300, bbox_inches="tight"); plt.show()
print(f"Figura: {p1}")

In [ ]:
# --- Comparativa 2: distribucion frecuencia-magnitud + Mc (MAXC) ---
def mc_maxc(mags, bw=0.1):
    bins = np.arange(np.floor(mags.min()*10)/10, np.ceil(mags.max()*10)/10 + bw, bw)
    counts, edges = np.histogram(mags, bins=bins)
    return float(edges[int(counts.argmax())] + bw/2)

mc_s = mc_maxc(ev_s["mag"].values)
mc_u = mc_maxc(ev_u["mag"].values)
print(f"Mc (MAXC, todas magnitudes)   STEAD = {mc_s:.2f}    USGS = {mc_u:.2f}")
print("Nota: el Mc bajo de STEAD refleja que incluye microsismos, NO que sea completo:")
print("      su curva G-R no sigue una recta limpia (sesgo de seleccion).")

mbins = np.arange(0, 8, 0.1)
cum_u = np.array([(ev_u["mag"].values >= m).sum() for m in mbins])
cum_s = np.array([(ev_s["mag"].values >= m).sum() for m in mbins])

fig, axs = plt.subplots(1, 2, figsize=(14, 5))
axs[0].hist(ev_u["mag"], bins=mbins, alpha=0.6, label="USGS", color="steelblue")
axs[0].hist(ev_s["mag"], bins=mbins, alpha=0.6, label="STEAD", color="darkred")
axs[0].axvline(MC_CMP, color="black", ls=":", lw=1.5, label=f"umbral pronostico M={MC_CMP}")
axs[0].set_xlabel("Magnitud"); axs[0].set_ylabel("n eventos")
axs[0].set_title("Histograma de magnitudes", fontweight="bold")
axs[0].legend(); axs[0].grid(alpha=0.3)
axs[1].semilogy(mbins, cum_u, "o-", ms=3, color="steelblue", label="USGS")
axs[1].semilogy(mbins, cum_s, "s-", ms=3, color="darkred",  label="STEAD")
axs[1].axvline(MC_CMP, color="black", ls=":", lw=1.5)
axs[1].set_xlabel("Magnitud"); axs[1].set_ylabel("N acumulado (>= M)")
axs[1].set_title("Distribucion Gutenberg-Richter", fontweight="bold")
axs[1].legend(); axs[1].grid(alpha=0.3)
plt.tight_layout()
p2 = os.path.join(FIGURES_DIR, f"{NB_TAG}_stead_vs_usgs_magnitud_mc.png")
plt.savefig(p2, dpi=300, bbox_inches="tight"); plt.show()
print(f"Figura: {p2}")

print()
print("="*70)
print("CONCLUSION (por que el pronostico usa USGS y no STEAD):")
print(f"  1. INCOMPLETO: a M>={MC_CMP}, USGS tiene {len(ev_u25)/max(len(ev_s25),1):.1f}x mas eventos que STEAD.")
print("  2. NO HOMOGENEO: los eventos/anio de STEAD oscilan ~100x de forma artificial,")
print("     asi que tasas, conteos y b-value calculados desde STEAD serian ruido.")
print("  3. ACABA EN 2018: no cubre el periodo de test (2023-2024) ni uso operacional.")
print("  => Las features de pronostico exigen catalogo completo+homogeneo+continuo = USGS.")
print("     STEAD (curado para picking) se reserva al notebook 02 (deteccion/picking).")
print("="*70)

## 7.2 Transferencia del Transformer a multiples regiones (test de generalizacion)

Se aplica el **Transformer entrenado en California+Alaska**, **en inferencia, sin reentrenar**, a
**10 regiones sismicas del mundo (2 por continente)** y se compara su AUC con la climatologia
local (Poisson) de cada region y con el **0.728 in-sample**.

Regiones: **Cascadia** y **Mexico** (N. America), **Chile** y **Peru** (S. America),
**Italia** y **Grecia** (Europa), **Japon** y **Sumatra** (Asia), **Rift Africano** y **Magreb** (Africa).

**Hipotesis:** el modelo **no generalizara** — su AUC caera por debajo del 0.728 in-sample y, sobre
todo, por debajo de la climatologia local, porque aprende una *funcion* ligada a la estadistica de
California+Alaska, no un patron universal (ya visto en Cascadia: AUC 0.577 < Poisson 0.809).

La celda recorre las 10 regiones replicando el pipeline del nb 05 y produce:
- una **grafica comun** (`07_transferencia_regiones.png`): AUC transferencia vs Poisson local por region,
- una **tabla de metricas** (`results/07_transferencia_metricas.csv`).

> **Caveats** (documentados): el *target* M>=4.5 en 3x3 / 30 d se construye **sin** declustering G-K, la
> normalizacion es per-region, y fuera de EE.UU. el Mc del USGS es mas alto (ver 7.0) — por eso la tabla
> **incluye el Mc** de cada region: parte de la caida puede deberse a menor completitud, no solo a la
> tectonica. La celda es **lenta en CPU** (descarga 10 regiones x 25 anios + inferencia con 5 checkpoints
> cada una): puede tardar varios minutos.

In [ ]:
# =====================================================================
# 7.2  Transferencia del Transformer a MULTIPLES regiones del mundo
# ---------------------------------------------------------------------
# Aplica el Transformer entrenado en California+Alaska, EN INFERENCIA (sin
# reentrenar), a 10 regiones sismicas (2 por continente). Mismo pipeline de
# features que el nb 05. Genera una GRAFICA COMUN y una TABLA de metricas.
# Caveats (documentados): target sin declustering G-K, normalizacion per-region,
# y fuera de EE.UU. el Mc del USGS es mas alto (ver 7.0) -> la tabla incluye el Mc.
# LENTO en CPU: descarga 10 regiones x 25 anios + inferencia con 5 checkpoints c/u.
import os, glob, sys
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
sys.path.append(os.path.abspath("."))
from utils.modelos import SpatioTemporalTransformer, geospatial_encoding
from utils.evaluacion import metricas_completas, auc_temporal_por_celda

def descargar_usgs(box, y0, y1, mmin=0):
    """Catalogo USGS (FDSN) ano por ano (evita el limite de 20000/peticion)."""
    frames = []
    for yr in range(y0, y1 + 1):
        url = ("https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv"
               f"&starttime={yr}-01-01&endtime={yr}-12-31"
               f"&minlatitude={box['minlat']}&maxlatitude={box['maxlat']}"
               f"&minlongitude={box['minlon']}&maxlongitude={box['maxlon']}"
               f"&minmagnitude={mmin}")
        try: frames.append(pd.read_csv(url))
        except Exception: pass
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def mc_maxc(mags, bw=0.1):
    bins = np.arange(np.floor(mags.min()*10)/10, np.ceil(mags.max()*10)/10 + bw, bw)
    c, e = np.histogram(mags, bins=bins); return float(e[int(c.argmax())] + bw/2)

# ---- Config identica al nb 05 ----
GRID_DEG, MIN_EV_CELL          = 0.5, 30
MC, M_TARGET                   = 2.5, 4.5
LOOKBACK, PRED_WIN, NEIGHBOR_KM = 60, 30, 100
D_MODEL, NUM_CH, KERNEL        = 128, (32, 32, 64), 3
N_HEADS, N_LAYERS, DIM_FF, DROP = 4, 2, 256, 0.20
AUC_INSAMPLE = 0.728            # AUC-ROC in-sample California+Alaska
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS_DIR = os.path.join("..", "results"); FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)
CKPTS = sorted(glob.glob(os.path.join(RESULTS_DIR, "05_ckpt_seed*.pt")))
assert CKPTS, "Faltan los checkpoints results/05_ckpt_seed*.pt"

# Cargar los 5 state_dicts UNA sola vez (se quita la mascara: depende de la region)
STATES = []
for ck in CKPTS:
    s = torch.load(ck, map_location=DEVICE, weights_only=False)
    if isinstance(s, dict) and "state_dict" in s: s = s["state_dict"]
    if isinstance(s, dict) and "model" in s: s = s["model"]
    STATES.append({k: v for k, v in s.items() if k != "attn_mask"})
print(f"device={DEVICE}  |  {len(STATES)} checkpoints cargados")

# ---- 10 regiones (2 por continente) ----
REGIONS = [
    {"name": "Cascadia (PNW)",   "cont": "N. America", "minlat": 40,  "maxlat": 49,  "minlon": -125, "maxlon": -119},
    {"name": "Mexico (Guerrero)","cont": "N. America", "minlat": 14,  "maxlat": 20,  "minlon": -102, "maxlon": -94},
    {"name": "Chile norte",      "cont": "S. America", "minlat": -35, "maxlat": -18, "minlon": -74,  "maxlon": -68},
    {"name": "Peru",             "cont": "S. America", "minlat": -18, "maxlat": -8,  "minlon": -78,  "maxlon": -70},
    {"name": "Italia (Apeninos)","cont": "Europa",     "minlat": 36,  "maxlat": 44,  "minlon": 12,   "maxlon": 18},
    {"name": "Grecia (Egeo)",    "cont": "Europa",     "minlat": 34,  "maxlat": 42,  "minlon": 19,   "maxlon": 28},
    {"name": "Japon",            "cont": "Asia",       "minlat": 30,  "maxlat": 46,  "minlon": 129,  "maxlon": 146},
    {"name": "Sumatra",          "cont": "Asia",       "minlat": -6,  "maxlat": 6,   "minlon": 94,   "maxlon": 104},
    {"name": "Rift E. Africano", "cont": "Africa",     "minlat": -5,  "maxlat": 12,  "minlon": 33,   "maxlon": 43},
    {"name": "Magreb",           "cont": "Africa",     "minlat": 30,  "maxlat": 37,  "minlon": -6,   "maxlon": 6},
]

def haversine(coords):
    R = 6371.0
    la = np.radians(coords[:, 0])[:, None]; lo = np.radians(coords[:, 1])[:, None]
    a = np.sin((la - la.T)/2)**2 + np.cos(la) * np.cos(la.T) * np.sin((lo - lo.T)/2)**2
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def evaluar_region(reg):
    """Replica el pipeline del nb 05 para 'reg' y devuelve sus metricas (o None)."""
    box = {k: reg[k] for k in ("minlat", "maxlat", "minlon", "maxlon")}
    cat = descargar_usgs(box, 2000, 2024, mmin=MC).dropna(subset=["mag"]).copy()
    if len(cat) < 200: return None
    cat["time"] = pd.to_datetime(cat["time"], errors="coerce", utc=True)
    cat = cat.dropna(subset=["time"]); cat["date"] = cat["time"].dt.floor("D")
    mc_reg = mc_maxc(cat["mag"].values)
    # grid + celdas activas
    lat_edges = np.arange(box["minlat"], box["maxlat"] + 1e-9, GRID_DEG)
    lon_edges = np.arange(box["minlon"], box["maxlon"] + 1e-9, GRID_DEG)
    n_lat, n_lon = len(lat_edges) - 1, len(lon_edges) - 1
    cat["lat_idx"] = np.clip(np.digitize(cat["latitude"],  lat_edges) - 1, 0, n_lat - 1)
    cat["lon_idx"] = np.clip(np.digitize(cat["longitude"], lon_edges) - 1, 0, n_lon - 1)
    cat["cell_id"] = cat["lat_idx"] * n_lon + cat["lon_idx"]
    epc = cat.groupby("cell_id").size()
    active = np.sort(epc[epc >= MIN_EV_CELL].index.values); n_active = len(active)
    if n_active < 6: return None
    cid2idx = {c: i for i, c in enumerate(active)}
    cat_a = cat[cat["cell_id"].isin(active)].copy(); cat_a["cell_idx"] = cat_a["cell_id"].map(cid2idx)
    cent = np.zeros((n_active, 2), np.float32); cell_ij = np.zeros((n_active, 2), np.float32)
    for i, c in enumerate(active):
        li, lo = c // n_lon, c % n_lon
        cent[i] = [(lat_edges[li] + lat_edges[li+1])/2, (lon_edges[lo] + lon_edges[lo+1])/2]
        cell_ij[i] = [li, lo]
    dist = haversine(cent); adj = ((dist <= NEIGHBOR_KM) & (dist > 0)).astype(np.float32)
    for i in range(n_active):
        if adj[i].sum() < 2:
            nn = np.argsort(dist[i]); nn = nn[nn != i][:2]; adj[i, nn] = 1; adj[nn, i] = 1
    attn_mask = torch.from_numpy((adj + np.eye(n_active)) == 0)
    pe = geospatial_encoding(cell_ij, D_MODEL).to(DEVICE)
    # features (5 canales)
    dr = pd.date_range("2000-01-01", "2024-12-31", freq="D", tz="UTC")
    n_days = len(dr); d2i = {d: i for i, d in enumerate(dr)}
    cat_a["day_idx"] = cat_a["date"].map(d2i); cat_a = cat_a.dropna(subset=["day_idx"])
    cat_a["day_idx"] = cat_a["day_idx"].astype(int); cat_a["energy"] = 10 ** (1.5 * cat_a["mag"] + 4.8)
    X = np.zeros((n_days, n_active, 5), np.float32); g = cat_a.groupby(["day_idx", "cell_idx"])
    v = g.size().reset_index(name="v");          X[v["day_idx"], v["cell_idx"], 0] = np.log1p(v["v"])
    v = g["energy"].sum().reset_index(name="v"); X[v["day_idx"], v["cell_idx"], 1] = np.log1p(v["v"])
    v = g["mag"].max().reset_index(name="v");    X[v["day_idx"], v["cell_idx"], 2] = v["v"].values
    d35 = cat_a[cat_a["mag"] >= 3.5].groupby(["day_idx", "cell_idx"]).size().reset_index(name="v")
    X[d35["day_idx"], d35["cell_idx"], 3] = np.log1p(d35["v"])
    mpd = (cat_a[cat_a["mag"] >= MC].groupby("date")["mag"].apply(list).reindex(dr)
             .apply(lambda x: x if isinstance(x, list) else []))
    bv = np.full(n_days, np.nan, np.float32)
    for i in range(n_days):
        win = [mm for sub in mpd.iloc[max(0, i-29):i+1] for mm in sub]
        if len(win) >= 20: bv[i] = np.log10(np.e) / (np.mean(win) - (MC - 0.05))
    X[:, :, 4] = pd.Series(bv, index=dr).ffill().fillna(1.0).values[:, None]
    mu, sd = X.reshape(-1, 5).mean(0), X.reshape(-1, 5).std(0) + 1e-8; Xn = (X - mu) / sd
    # target 3x3 / 30d (sin declustering)
    tgt = cat[cat["mag"] >= M_TARGET]; n_tgt = len(tgt)
    ms = np.zeros((n_days, n_active), bool)
    for _, r in tgt.iterrows():
        di = d2i.get(r["date"])
        if di is None: continue
        li, lo = int(r["lat_idx"]), int(r["lon_idx"])
        for a in (-1, 0, 1):
            for b_ in (-1, 0, 1):
                cid = (li + a) * n_lon + (lo + b_)
                if cid in cid2idx: ms[di, cid2idx[cid]] = True
    y = np.zeros((n_days, n_active), np.int8)
    for t in range(n_days - PRED_WIN): y[t] = ms[t+1:t+1+PRED_WIN].any(0)
    valid = list(range(LOOKBACK, n_days - PRED_WIN)); y_ev = y[valid]
    if y_ev.sum() < 5: return None
    # inferencia ensemble
    probs = np.zeros((len(valid), n_active), np.float32)
    with torch.no_grad():
        for st in STATES:
            model = SpatioTemporalTransformer(n_inputs=5, num_channels=NUM_CH, kernel_size=KERNEL,
                    d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS, dropout=DROP,
                    dim_feedforward=DIM_FF, attn_mask=attn_mask).to(DEVICE)
            model.load_state_dict(st, strict=False); model.eval()
            out = []
            for i in range(0, len(valid), 64):
                ts = valid[i:i+64]
                xb = np.stack([Xn[t-LOOKBACK:t] for t in ts]).transpose(0, 2, 1, 3)
                out.append(torch.sigmoid(model(torch.from_numpy(xb).float().to(DEVICE), pe)).cpu().numpy())
            probs += np.concatenate(out, 0) / len(STATES)
    yf, pf = y_ev.reshape(-1), probs.reshape(-1)
    m = metricas_completas(yf, pf); mP = metricas_completas(yf, np.tile(y_ev.mean(0), len(valid)))
    at, _ = auc_temporal_por_celda(yf, pf, n_active, min_pos=5)
    return {"region": reg["name"], "continente": reg["cont"], "Mc": mc_reg, "celdas": n_active,
            "n_target": int(n_tgt), "auc_modelo": m["roc_auc"], "auc_poisson": mP["roc_auc"],
            "pr_auc": m["pr_auc"], "lift1": m["lift@1%"], "auc_temporal": at}

# ---- Bucle sobre las regiones ----
res = []
for reg in REGIONS:
    print(f"[{reg['name']:>18}] ...", end=" ", flush=True)
    try:
        r = evaluar_region(reg)
        if r is None:
            print("descartada (pocos datos/celdas/targets)"); continue
        res.append(r)
        print(f"AUC={r['auc_modelo']:.3f}  Poisson={r['auc_poisson']:.3f}  "
              f"celdas={r['celdas']}  Mc={r['Mc']:.1f}")
    except Exception as e:
        print(f"error: {type(e).__name__}: {str(e)[:60]}")

df = pd.DataFrame(res)
csv_path = os.path.join(RESULTS_DIR, "07_transferencia_metricas.csv")
df.to_csv(csv_path, index=False)

print("\n=== TABLA DE TRANSFERENCIA (Transformer CA+Alaska -> otras regiones) ===")
print(df.to_string(index=False, float_format=lambda x: f"{x:.3f}",
      columns=["region", "continente", "Mc", "celdas", "n_target",
               "auc_modelo", "auc_poisson", "lift1", "auc_temporal"]))
print(f"\nCSV: {csv_path}")

# ---- GRAFICA COMUN: AUC transferencia vs climatologia local, por region ----
d = df.sort_values("auc_modelo").reset_index(drop=True)
yp = np.arange(len(d)); h = 0.38
fig, ax = plt.subplots(figsize=(10, 0.62 * len(d) + 2))
ax.barh(yp + h/2, d["auc_modelo"],  height=h, color="#C44E52", label="Transformer (transferencia)")
ax.barh(yp - h/2, d["auc_poisson"], height=h, color="#8172B3", label="Poisson local (climatologia)")
ax.axvline(0.5, color="k", ls=":", lw=1, label="azar (0.5)")
ax.axvline(AUC_INSAMPLE, color="green", ls="--", lw=1.6, label=f"in-sample CA+AK ({AUC_INSAMPLE})")
ax.set_yticks(yp); ax.set_yticklabels([f"{r}  [{c}]" for r, c in zip(d["region"], d["continente"])])
ax.set_xlabel("AUC-ROC"); ax.set_xlim(0.3, 1.0)
ax.set_title("Transferencia del Transformer (entrenado en California+Alaska) a otras regiones",
             fontweight="bold")
ax.legend(loc="lower right"); ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "07_transferencia_regiones.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight"); plt.show()
print(f"Figura: {fig_path}")

# ---- Sintesis ----
peor_clima = (df["auc_modelo"] < df["auc_poisson"]).mean() * 100
print("\n" + "=" * 70)
print(f"En {peor_clima:.0f}% de las regiones el modelo transferido NO supera a la climatologia local.")
print(f"AUC medio de transferencia: {df['auc_modelo'].mean():.3f}   (vs 0.728 in-sample CA+Alaska).")
print(f"AUC temporal/celda medio:   {df['auc_temporal'].mean():.3f}  (~0.5 = sin senal temporal, como en casa).")
print("=> El modelo cartografia bien el peligro DONDE se entrena, pero NO transfiere a otras regiones.")
print("=" * 70)